In [82]:
import torch
import torch.nn as nn


class TwoTensorMPS(nn.Module):
    def __init__(self, physical_dim=2, bond_dim=3):
        super().__init__()

        self.A1 = nn.Parameter(torch.randn(physical_dim, bond_dim) * 0.1)
        self.A2 = nn.Parameter(torch.randn(bond_dim, physical_dim) * 0.1)

    def forward(self, x):
        """
        x shape: [batch, 2]
        x[:, 0] = first input
        x[:, 1] = second input
        """

        x1 = x[:,0]
        x2 = x[:,1].T

        # Select slices
        left = x1 @ self.A1     # [batch, bond_dim]
        right = self.A2 @ x2  # [batch, bond_dim]

        # Contract bond
        y = torch.sum(left @ right)

        return y

In [ ]:
model = TwoTensorMPS(physical_dim=2, bond_dim=3)

x = torch.tensor([
    [[1.0, 0.0], [0.0, 1.0]],
    [[1,2],[3,4]]
])


y = model.forward(x)
print(y)


tensor([[0., 3.],
        [1., 4.]])
tensor(0.1404, grad_fn=<SumBackward0>)


In [86]:
target = torch.tensor([1])  

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(100):
    pred = model.forward(x)
    loss = torch.mean((pred - target) ** 2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(model(x))

tensor(0.9964, grad_fn=<SumBackward0>)
